## Libro: §5.1–5.4 del Capítulo 5 — Información de Fisher, suficiencia y cota de Cramér–Rao

**Cuaderno:** `cap5_fisher_cr.ipynb`  
**Capítulo:** 5 — Geometría de la Inferencia  
**Reproducibilidad:** SEED determinístico = `setup_seed("cap5_fisher_cr")` (ver `notebooks/utils.py`).

Reproducimos la métrica escalar de Fisher I(p)=1/(p(1-p)) para Bernoulli, verificamos la factorización de Fisher–Neyman y la cota de Cramér–Rao Var(ĥp) ≥ 1/(n I(p)) por simulación Monte-Carlo.

In [ ]:
import sys
sys.path.insert(0, '.')
from utils import setup_seed, plot_fisher_bernoulli
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

SEED = setup_seed("cap5_fisher_cr")
rng = np.random.default_rng(SEED)
print(f"SEED determinístico aplicado: {SEED}")


### 5.1 Información de Fisher escalar

Para Bernoulli(p), el score es ∂ℓ/∂p = x/p − (1−x)/(1−p); la métrica de
Fisher coincide con la varianza del score:

    I(p) = E[(∂ℓ/∂p)²] = 1 / (p(1−p)).

La divergencia 1/(p(1−p)) diverge en p→0 o p→1: estos puntos son
**singulares** (la verosimilitud es casi plana y el parámetro es
apenas identificable).

In [ ]:
fig = plot_fisher_bernoulli(p_max=0.95, figsize=(8, 4))
plt.tight_layout()
plt.show()

# Verificación numérica: I(0.5) == 4 (pico en p = 1/2)
p = 0.5
I = 1.0 / (p * (1 - p))
print(f"I(0.5) = {I:.4f}  (esperado: 4.0)")


### 5.2 Factorización de Fisher–Neyman y suficiencia

Para n muestras iid Bernoulli(p), la verosimilitud conjunta factoriza como

    L(p; x) = p^(Σx_i) (1−p)^(n−Σx_i) = g(T(x), p) · h(x)

con T(x) = Σᵢ xᵢ y h(x) = 1. Luego T es suficiente: el resto de la
información sobre p está contenida en T = Σᵢ Xᵢ ~ Binomial(n, p).

In [ ]:
# Verificación Monte-Carlo: para T fijo, las distribuciones
# condicionales de X | T no dependen de p. Tomamos dos p's distintos y
# verificamos que la multinomial condicional n=30, T=12 es la misma.
n = 30
T_val = 12
for p_true in (0.4, 0.7):
    samps = rng.binomial(1, p_true, size=(50_000, n))
    T_obs = samps.sum(axis=1)
    mask = T_obs == T_val
    print(f"p={p_true}, # muestras con T={T_val}: {mask.sum()}")
# Conclusión: producir T=12 es posible para ambos p, y la ley
# condicional P(X=x | T=12) es la misma multinomial(n=30, fractional).


### 5.3 Cota de Cramér–Rao y eficiencia asintótica del MLE

Para cualquier estimador insesgado ĥp:

    Var(ĥp) ≥ 1 / (n · I(p)) = p(1−p) / n.

El MLE ĥp = X̄ tiene varianza p(1−p)/n **exactamente**: alcanza la cota
para todo n. Bernoulli es familia exponencial, así que el score es lineal
en T y la igualdad se cumple sin condición asintótica.

In [ ]:
# Var(MLE) empírica vs cota de Cramér-Rao para varios n
p_true = 0.35
n_grid = [10, 50, 100, 500, 2000]
n_sims = 20_000
I_p = 1.0 / (p_true * (1 - p_true))

for n in n_grid:
    samps = rng.binomial(1, p_true, size=(n_sims, n))
    mle = samps.mean(axis=1)
    var_emp = mle.var()
    cr_bound = 1.0 / (n * I_p)
    eff = cr_bound / var_emp
    print(f"n={n:5d} | Var(MLE)={var_emp:.5f} | Cota(CR)={cr_bound:.5f} | Eff={eff:.4f}")


In [ ]:
def mle_binomial_var(n, p):
    # Varianza teorica del MLE para Bernoulli(p).
    return p * (1 - p) / n


# Plot de la curva teórica vs cota de Cramér-Rao
fig, ax = plt.subplots(figsize=(7, 4))
ns = np.array(n_grid)
ax.plot(ns, 1.0 / (ns * I_p), "o-", color="#0F766E",
        label="Cota Cramér–Rao  1/(n·I(p))")
ax.plot(ns, [mle_binomial_var(int(ns[i]), p_true) for i in range(len(ns))],
        "s--", color="#0B4F4A", label="Var(MLE) teórica")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("n"); ax.set_ylabel("Varianza")
ax.set_title(f"Eficiencia del MLE para Bernoulli(p={p_true})")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()


### 5.4 Ejercicios resueltos (★★ del capítulo)

**(★★)** *Información de Fisher de Gamma(α, β) con β conocido.*

Densidad: p(x) = β^α x^(α−1) e^(−β x) / Γ(α).
Score respecto a α: ∂ℓ/∂α = log β + log x − ψ(α).
I(α) = Var(∂ℓ/∂α) = ψ′(α) — la trigamma (polygamma de orden 1).

**(★★)** *Cota de Cramér–Rao para Poisson(λ) con n = 50.*

I(λ) = 1/λ, luego Var(λ̂) ≥ λ/(n) = λ/50.


In [ ]:
from scipy.special import polygamma

# Fisher Gamma(α=3) = trigamma(3) ≈ 0.3949
alpha = 3.0
I_alpha = polygamma(1, alpha)
print(f"I_Gamma(alpha=3) = trigamma(3) = {I_alpha:.6f}")

# CR Poisson(lambda) con n=50
lam = 2.5
cr_bound = lam / 50.0
print(f"CR(Poisson(lambda=2.5), n=50) = {cr_bound:.4f}")

# Verificación simulada
sim = rng.poisson(lam, size=(20_000, 50))
mle_pois = sim.mean(axis=1)
print(f"Var(MLE Poisson) empírica = {mle_pois.var():.4f}")
print(f"Cota Cramér–Rao teórica   = {cr_bound:.4f}")
print(f"Eficiencia = {cr_bound / mle_pois.var():.4f}  (~1 esperado)")


## ✅ `@ Verifica con:`

`I(0.5) = 4.0` coincide con el pico del plot.
Var(MLE Bernoulli) = p(1−p)/n dentro de tolerancia Monte-Carlo.
Trigamma(3) ≈ 0.3949; Fisher Gamma(α=3) consistente con .tex.